In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timedelta

headers = {
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/json; charset=utf-8",
    "Cookie": " ",
    "User-Agent": " "
}

In [ ]:
import os
from pathlib import Path

def find_repo_root(marker: str = 'data') -> Path:
    """Tìm thư mục gốc của repo dựa trên marker."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError('Không tìm thấy repo root.')

ROOT = find_repo_root()
os.chdir(ROOT)
print(f' Working directory: {ROOT}')


# Thu thập dữ liệu phụ tải và giá biên từ NSMO

Notebook này thực hiện thu thập dữ liệu từ API của NSMO cho hai chỉ số chính: Phụ tải (Load) và Giá biên (Price). 

**Chi tiết thu thập:**
- **Nguồn:** NSMO API (`GetChartPhuTaiVM` và `GetChartGiaBienVM`).
- **Phạm vi:** Từ 11/03/2023 đến 10/03/2026.
- **Tần suất:** 30 phút/mẫu.
- **Sản phẩm:** Các file CSV lưu theo batch 40 ngày tại `data/raw/NSMO/`.

In [ ]:
def crawl_nsmo_sync(start_limit_str, end_limit_str):
    total_start = datetime.strptime(start_limit_str, "%Y-%m-%d")
    total_end = datetime.strptime(end_limit_str, "%Y-%m-%d")
    output_dir = 'data/raw/NSMO'
    if not os.path.exists(output_dir): 
        os.makedirs(output_dir)

    current_end = total_end
    
    while current_end >= total_start:
        current_start = current_end - timedelta(days=40)
        if current_start < total_start: 
            current_start = total_start
        
        s_str = current_start.strftime('%Y-%m-%d')
        e_str = current_end.strftime('%Y-%m-%d')
        
        file_name = f"NSMO_{s_str}_to_{e_str}.csv"
        file_path = os.path.join(output_dir, file_name)
        
        if os.path.exists(file_path):
            print(f"Skipping existing file: {file_name}")
        else:
            print(f"Processing batch: {s_str} -> {e_str}")
            all_days_data = []
            curr = current_start
            
            while curr <= current_end:
                day_query = curr.strftime("%d/%m/%Y").replace("/", "%2F")
                
                try:
                    url_load = f"https://www.nsmo.vn/api/services/app/Pages/GetChartPhuTaiVM?day={day_query}"
                    url_price = f"https://www.nsmo.vn/api/services/app/Pages/GetChartGiaBienVM?day={day_query}"
                    
                    res_load = requests.get(url_load, headers=headers, timeout=15).json()
                    res_price = requests.get(url_price, headers=headers, timeout=15).json()
                    
                    data_load = res_load['result']['data']['phuTais']
                    data_price = res_price['result']['data']['giaBiens']
                    
                    if data_load and data_price:
                        df_load = pd.DataFrame(data_load)
                        df_price = pd.DataFrame(data_price)
                        
                        df_day = pd.merge(df_load, df_price, on='thoiGian', how='outer')
                        
                        df_day = df_day.rename(columns={
                            'thoiGian': 'Timestamp',
                            'congSuatHT': 'Load_National', 'congSuatMB': 'Load_North', 
                            'congSuatMT': 'Load_Central', 'congSuatMN': 'Load_South',
                            'giaBienHT': 'Price_National', 'giaBienMB': 'Price_North', 
                            'giaBienMT': 'Price_Central', 'giaBienMN': 'Price_South'
                        })
                        all_days_data.append(df_day)
                    
                    time.sleep(0.5)
                except Exception as e:
                    print(f"Error at {curr.strftime('%Y-%m-%d')}: {e}")
                
                curr += timedelta(days=1)
            
            if all_days_data:
                df_batch = pd.concat(all_days_data, ignore_index=True)
                df_batch.to_csv(file_path, index=False, encoding='utf-8-sig')
                print(f"Saved: {file_name}")
        
        current_end = current_start - timedelta(days=1)
        time.sleep(2)

**Logic xử lý:**
1. **Chia Batch:** Dữ liệu được chia thành các batch khoảng 40 ngày để tránh file quá lớn và dễ quản lý khi có lỗi xảy ra.
2. **Merge dữ liệu:** Sử dụng `outer merge` trên cột `thoiGian` để đảm bảo không mất dữ liệu khi một trong hai API (Load hoặc Price) bị thiếu mẫu tại một thời điểm.
3. **Xử lý lỗi:** Chương trình tự động bỏ qua các ngày không có dữ liệu và tiếp tục với ngày tiếp theo.

In [ ]:
crawl_nsmo_sync("2023-03-11", "2026-03-10")